In [7]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

PROCESSED_PATH = os.path.join("..", "dataset", "processed")

print("Processed path:", os.path.abspath(PROCESSED_PATH))

Processed path: c:\Users\palak\SANKETSETU\dataset\processed


In [8]:
records = []

for category in sorted(os.listdir(PROCESSED_PATH)):
    category_path = os.path.join(PROCESSED_PATH, category)

    if not os.path.isdir(category_path):
        continue

    for sign in sorted(os.listdir(category_path)):
        sign_path = os.path.join(category_path, sign)

        if not os.path.isdir(sign_path):
            continue

        for file in os.listdir(sign_path):
            if file.endswith(".npy"):
                records.append({
                    "Category": category,
                    "Sign": sign,
                    "File": file,
                    "Path": os.path.join(sign_path, file)
                })

dataset_df = pd.DataFrame(records)

print("Total processed videos:", len(dataset_df))
dataset_df.head()

Total processed videos: 3652


,Category,Sign,File,Path
0,Adjectives,1. loud,MVI_5177.npy,..\dataset\processed\Adjectives\1. loud\MVI_51...
1,Adjectives,1. loud,MVI_5178.npy,..\dataset\processed\Adjectives\1. loud\MVI_51...
2,Adjectives,1. loud,MVI_5179.npy,..\dataset\processed\Adjectives\1. loud\MVI_51...
3,Adjectives,1. loud,MVI_5257.npy,..\dataset\processed\Adjectives\1. loud\MVI_52...
4,Adjectives,1. loud,MVI_5258.npy,..\dataset\processed\Adjectives\1. loud\MVI_52...


In [9]:
labels = sorted(dataset_df["Sign"].unique())

label_to_id = {label: idx for idx, label in enumerate(labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

dataset_df["Label"] = dataset_df["Sign"].map(label_to_id)

print("Total unique labels:", len(label_to_id))
dataset_df.head()

Total unique labels: 262


,Category,Sign,File,Path,Label
0,Adjectives,1. loud,MVI_5177.npy,..\dataset\processed\Adjectives\1. loud\MVI_51...,2
1,Adjectives,1. loud,MVI_5178.npy,..\dataset\processed\Adjectives\1. loud\MVI_51...,2
2,Adjectives,1. loud,MVI_5179.npy,..\dataset\processed\Adjectives\1. loud\MVI_51...,2
3,Adjectives,1. loud,MVI_5257.npy,..\dataset\processed\Adjectives\1. loud\MVI_52...,2
4,Adjectives,1. loud,MVI_5258.npy,..\dataset\processed\Adjectives\1. loud\MVI_52...,2


In [10]:
lengths = []

for path in tqdm(dataset_df["Path"]):
    arr = np.load(path)
    lengths.append(arr.shape[0])

dataset_df["Frames"] = lengths

print(dataset_df["Frames"].describe())
print("Maximum frames:", dataset_df["Frames"].max())
print("Minimum frames:", dataset_df["Frames"].min())

100%|██████████| 3652/3652 [00:00<00:00, 4560.30it/s]

count    3652.000000
mean       62.965225
std        14.872715
min        33.000000
25%        53.000000
50%        60.000000
75%        70.000000
max       154.000000
Name: Frames, dtype: float64
Maximum frames: 154
Minimum frames: 33


In [11]:
MAX_FRAMES = dataset_df["Frames"].max()

print("Using max sequence length:", MAX_FRAMES)

def pad_sequence(sequence, max_len):
    padded = np.zeros((max_len, 63), dtype=np.float32)

    length = min(sequence.shape[0], max_len)
    padded[:length] = sequence[:length]

    return padded

Using max sequence length: 154


In [12]:
X = []
y = []

for _, row in tqdm(dataset_df.iterrows(), total=len(dataset_df)):
    sequence = np.load(row["Path"])
    sequence = pad_sequence(sequence, MAX_FRAMES)

    X.append(sequence)
    y.append(row["Label"])

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

100%|██████████| 3652/3652 [00:01<00:00, 2623.97it/s]


X shape: (3652, 154, 63)
y shape: (3652,)


In [14]:
from sklearn.model_selection import train_test_split

# Split without stratify because some classes have only one sample
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (2921, 154, 63)
Validation: (365, 154, 63)
Test: (366, 154, 63)


In [15]:
TRAIN_PATH = os.path.join("..", "dataset", "training")
os.makedirs(TRAIN_PATH, exist_ok=True)

np.save(os.path.join(TRAIN_PATH, "X_train.npy"), X_train)
np.save(os.path.join(TRAIN_PATH, "X_val.npy"), X_val)
np.save(os.path.join(TRAIN_PATH, "X_test.npy"), X_test)

np.save(os.path.join(TRAIN_PATH, "y_train.npy"), y_train)
np.save(os.path.join(TRAIN_PATH, "y_val.npy"), y_val)
np.save(os.path.join(TRAIN_PATH, "y_test.npy"), y_test)

print("✅ Training dataset saved successfully!")
print("Saved at:", os.path.abspath(TRAIN_PATH))

✅ Training dataset saved successfully!
Saved at: c:\Users\palak\SANKETSETU\dataset\training
